In [ ]:
# === CHALLENGE 2 (TNF-alpha) : PRODUCTION SETUP ===
# Builds the CROPPED target. The smoke test measured 43.3 s per iteration on the full
# two-chain construct (304 + 70 residues) - 7.2x Challenge 1's rate for a 1.7x length
# increase, so the cost scales nearer L^3.7 than L^2. At that rate the whole GPU budget
# buys about six designs, which is not worth running.
#
# Cropping to a 22 A sphere around the switch site keeps all 25 site residues, both
# switch anchors (K166 of chain A, R108 of chain B) and - measured against the intact
# trimer - introduces ZERO artificially exposed surface at the site, fewer than the
# uncropped two-chain construct, because cropping also removes residues that chain C
# had been covering.
import glob, json, os, shutil, subprocess, sys, time
t0 = time.time()
OFF = 76
RADIUS = 22.0


def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd)
        print((r.stdout + r.stderr)[-1000:])
        if check:
            raise SystemExit("setup failed")
    return r


print("python", sys.version.split()[0], flush=True)
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip(), flush=True)

os.makedirs("/kaggle/working/params", exist_ok=True)
src = glob.glob("/kaggle/input/**/params/params_model_*.npz", recursive=True)
if src:
    print("reusing %d AF2 param files from the attached input" % len(src), flush=True)
    for f in src:
        dst = "/kaggle/working/params/" + os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy(f, dst)
else:
    print("downloading AF2 params (~3.5 GB) ...", flush=True)
    sh("wget -q -O /kaggle/working/af2.tar "
       "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
    sh("tar -xf /kaggle/working/af2.tar -C /kaggle/working/params")
    sh("rm -f /kaggle/working/af2.tar")
print("AF2 params present:", len(glob.glob("/kaggle/working/params/*.npz")), "files", flush=True)

sh("pip install -q biopython")
import urllib.request
import numpy as np
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import protein_letters_3to1

os.makedirs("/kaggle/working/targets", exist_ok=True)
raw = "/kaggle/working/1TNF.pdb"
if not os.path.exists(raw):
    open(raw, "wb").write(urllib.request.urlopen(
        "https://files.rcsb.org/download/1TNF.pdb", timeout=180).read())

P = PDBParser(QUIET=True)
mdl = P.get_structure("m", raw)[0]
ca = {}
for ch in mdl:
    for r in ch:
        if r.id[0] == " " and "CA" in r:
            ca[(ch.id, r.id[1])] = r["CA"].get_coord()
centre = (ca[("A", 166 - OFF)] + ca[("B", 108 - OFF)]) / 2
keep = {k for k, v in ca.items() if k[0] in ("A", "B")
        and float(np.linalg.norm(v - centre)) <= RADIUS}


class Crop(Select):
    def accept_chain(self, c): return c.id in ("A", "B")
    def accept_residue(self, r): return r.id[0] == " " and (r.parent.id, r.id[1]) in keep
    def accept_atom(self, a): return a.element != "H"


TARGET = "/kaggle/working/targets/TNF_site_crop.pdb"
st = P.get_structure("t", raw)
io = PDBIO(); io.set_structure(st); io.save(TARGET, Crop())

# The map from construct order to the original chain and UniProt number. AfDesign
# CONCATENATES the target chains into one chain on output and renumbers from 1 - the
# smoke test showed chains {'A': 608, 'B': 140} where three chains were expected - so
# position in this list IS the output residue number minus one. Everything downstream
# maps through this rather than assuming a numbering.
order = []
for ch in P.get_structure("c", TARGET)[0]:
    for r in ch:
        if r.id[0] == " ":
            order.append(dict(chain=ch.id, pdb=r.id[1], uniprot=r.id[1] + OFF,
                              aa=protein_letters_3to1.get(r.get_resname(), "X")))
json.dump(dict(radius=RADIUS, n=len(order), order=order),
          open("/kaggle/working/construct_map.json", "w"))

SITE = [95, 105, 106, 107, 108, 109, 110, 111, 112, 155, 159, 161, 162, 163, 164, 165,
        166, 167, 168, 201, 203, 207, 220, 221, 223]
have = {(o["chain"], o["uniprot"]) for o in order}
print("cropped target: %d residues (A %d, B %d)"
      % (len(order), sum(1 for o in order if o["chain"] == "A"),
         sum(1 for o in order if o["chain"] == "B")), flush=True)
print("site residues retained: %d of %d"
      % (len({o["uniprot"] for o in order} & set(SITE)), len(SITE)), flush=True)
assert ("A", 166) in have and ("B", 108) in have, "a switch anchor was cropped away"
assert len({o["uniprot"] for o in order} & set(SITE)) == len(SITE), "site residue lost"
print("switch anchors present: K166(A), R108(B)", flush=True)

HOTSPOT = "A86,A87,A88,A90,A91,A92,B29,B32,B33,B34,B35,B36"

PROBE = ("python -c \"import jax, jax.lib; assert hasattr(jax.lib,'xla_bridge'); "
         "print(jax.__version__)\"")
r = subprocess.run(PROBE, shell=True, capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip().startswith("0.4.38"):
    print("installing jax 0.4.38 ...", flush=True)
    sh("pip install -q 'jax[cuda12]==0.4.38' 'jaxlib==0.4.38'")
sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git")

chk = subprocess.run(
    "cd /kaggle/working && python -c \""
    "import jax; from colabdesign import mk_afdesign_model; "
    "m = mk_afdesign_model(protocol='binder', data_dir='/kaggle/working'); "
    "m.prep_inputs(pdb_filename='%s', chain='A,B', binder_len=65, hotspot='%s'); "
    "print('READY', jax.__version__, jax.devices()); "
    "print('target length seen by the model:', m._target_len)\"" % (TARGET, HOTSPOT),
    shell=True, capture_output=True, text=True)
print(chk.stdout[-500:])
if chk.returncode != 0:
    print("PREP FAILED:"); print(chk.stderr[-1500:])
print("TNF PRODUCTION SETUP DONE in %.1f min" % ((time.time() - t0) / 60))

In [ ]:
# === CHALLENGE 2 (TNF-alpha) : CONFIRMATION OF THE SUBMITTED DESIGNS, ROUND 2 ===
#
# The switch stage scored each variant by prediction but did not save the predicted
# complex, so two things were asserted rather than measured:
#
#   1. that a switch variant still binds the SAME epitope as its parent. Retaining
#      i_ptm does not prove that - a histidine could shift the binding mode and keep a
#      similar score. On EGFR the designs drifted off the intended patch while looking
#      fine, and that was only found at the end.
#   2. that each designed histidine actually ends up facing the cation it was placed
#      against. Placement was computed on the PARENT backbone; whether the geometry
#      survives re-folding with three residues changed is a separate question, and it
#      is the one the whole acid-OFF hypothesis rests on.
#
# This re-predicts the 16 submitted sequences WITH structures saved and measures both,
# plus the control for each pair, so any epitope shift can be attributed to the
# histidines rather than to prediction variance between two runs.
import glob, json, os, subprocess, sys, textwrap, time

TIME_BUDGET_MIN = 60

WORKER = r"""
import glob, json, os, sys, time
from collections import Counter
GPU, SHARD, BUDGET = sys.argv[1], int(sys.argv[2]), float(sys.argv[3])
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
t0 = time.time()
def log(m): print("[w%d] %s [%.1f min]" % (SHARD, m, (time.time()-t0)/60), flush=True)

import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import protein_letters_3to1

TARGET = "/kaggle/working/targets/TNF_site_crop.pdb"
CMAP = json.load(open("/kaggle/working/construct_map.json"))["order"]
N = len(CMAP)
SITE = {95,105,106,107,108,109,110,111,112,155,159,161,162,163,164,165,166,167,168,
        201,203,207,220,221,223}
CAT_N = {"ARG": ("NE","NH1","NH2"), "LYS": ("NZ",)}
# index into the construct for each partner named in the switch record, e.g. "R108B"
PIDX = {}
for i, o in enumerate(CMAP):
    PIDX["%s%d%s" % (o["aa"], o["uniprot"], o["chain"])] = i

def vcb(r):
    if "CB" in r: return r["CB"].get_coord()
    try: n,a,c = r["N"].get_coord(), r["CA"].get_coord(), r["C"].get_coord()
    except KeyError: return None
    b, cc = a-n, c-a
    return -0.58273431*np.cross(b,cc) + 0.56802827*b - 0.54067466*cc + a

def trim(chain, n, what):
    res = [r for r in chain if r.id[0] == " "]
    if len(res) > n and len(res) % n == 0:
        for r in res[n:]:
            chain.detach_child(r.id)
        res = res[:n]
    if len(res) != n:
        raise ValueError("%s chain has %d residues, expected %d" % (what, len(res), n))
    return res

def measure(pdb_path, blen):
    mdl = PDBParser(QUIET=True).get_structure("c", pdb_path)[0]
    sizes = Counter({ch.id: sum(1 for r in ch if r.id[0]==" ") for ch in mdl})
    tgt, bnd = max(sizes, key=lambda k: sizes[k]), min(sizes, key=lambda k: sizes[k])
    tres = trim(mdl[tgt], N, "target")
    bres = trim(mdl[bnd], blen, "binder")
    seen = "".join(protein_letters_3to1.get(r.get_resname(), "X") for r in tres)
    if seen != "".join(o["aa"] for o in CMAP):
        raise ValueError("target sequence does not match construct_map")
    batoms = np.array([a.get_coord() for r in bres for a in r])
    epi = []
    for i, r in enumerate(tres):
        ra = np.array([a.get_coord() for a in r])
        if np.min(np.linalg.norm(ra[:,None,:]-batoms[None,:,:], axis=-1)) <= 5.0:
            epi.append(CMAP[i]["uniprot"])
    return sorted(set(epi)), tres, bres

def his_contacts(tres, bres, his_pos, partners):
    # closest Cbeta->(cation N) distance for each designed histidine, against the
    # partners it was placed to face. Same measurement the placement used.
    out = []
    for p in his_pos:
        r = bres[p-1]
        cb = vcb(r)
        aa = protein_letters_3to1.get(r.get_resname(), "X")
        best, who = 99.0, "-"
        for name in partners:
            tr = tres[PIDX[name]]
            at = [tr[a].get_coord() for a in CAT_N.get(tr.get_resname(), ()) if a in tr]
            for x in at:
                d = float(np.linalg.norm(cb - x))
                if d < best:
                    best, who = d, name
        out.append((p, aa, who, round(best, 2)))
    return out

from colabdesign import mk_afdesign_model, clear_mem
HOTSPOT = "A86,A87,A88,A90,A91,A92,B29,B32,B33,B34,B35,B36"
_c = {}
def predict(seq, out_pdb):
    if _c.get("L") != len(seq):
        clear_mem()
        af = mk_afdesign_model(protocol="binder", data_dir="/kaggle/working")
        af.prep_inputs(pdb_filename=TARGET, chain="A,B", binder_len=len(seq),
                       hotspot=HOTSPOT)
        _c["L"], _c["af"] = len(seq), af
    af = _c["af"]
    try:
        af.predict(seq=seq, num_recycles=3, verbose=False)
    except TypeError:
        af.predict(seq=seq, verbose=False)
    lg = af.aux["log"]
    af.save_pdb(out_pdb)
    return (round(float(lg.get("i_ptm", lg.get("ptm", 0.0))), 3),
            round(float(lg.get("plddt", 0.0)), 3))

rj = glob.glob("/kaggle/input/**/tnf_designs_final2.json", recursive=True)
if not rj:
    raise SystemExit("tnf_designs_final2.json not found - attach the tnf_switch output")
recs = {r["name"]: r for r in json.load(open(rj[0]))}
usable = sorted([r for r in recs.values()
                 if r["role"] == "switch" and r["i_ptm"] >= 0.45
                 and r["iptm_retained"] >= 0.70],
                key=lambda r: -r["i_ptm"])
# pairs stay on one worker: the control shares its switch's length, so the model is
# built once per pair instead of twice
mine = [r for i, r in enumerate(usable) if i % 2 == SHARD]
if SHARD == 0:
    log("%d records, %d usable switches to confirm" % (len(recs), len(usable)))
log("this worker takes %d: %s" % (len(mine), ", ".join(r["name"] for r in mine)))

os.makedirs("/kaggle/working/confirm2", exist_ok=True)
rec_path = "/kaggle/working/confirm2_w%d.jsonl" % SHARD
for sw in mine:
    if (time.time()-t0)/60 > BUDGET:
        log("time budget reached"); break
    ctl = recs.get(sw["source_backbone"] + "_ctl")
    parent_epi = set(sw["epitope_residues"])
    his_pos = [int(x[1:-1]) for x in sw["his_substitutions"].split(",")]
    row = dict(name=sw["name"], source=sw["source_backbone"], length=sw["length"],
               his_substitutions=sw["his_substitutions"],
               intended_partners=sw["his_partners"],
               stage3_i_ptm=sw["i_ptm"], parent_epitope=sorted(parent_epi))
    try:
        for tag, r in (("sw", sw), ("ctl", ctl)):
            if r is None:
                continue
            out = "/kaggle/working/confirm2/%s_%s.pdb" % (sw["source_backbone"], tag)
            ip, pl = predict(r["sequence"], out)
            epi, tres, bres = measure(out, r["length"])
            keep = sorted(set(epi) & parent_epi)
            row[tag] = dict(i_ptm=ip, plddt=pl, epitope=epi,
                            on_site=sorted(set(epi) & SITE),
                            kept_parent=len(keep),
                            frac_parent=round(len(keep)/max(len(parent_epi), 1), 3))
            if tag == "sw":
                row["his_geometry"] = his_contacts(tres, bres, his_pos,
                                                   sw["his_partners"])
        g = row.get("his_geometry", [])
        near = sum(1 for _, _, _, d in g if d <= 8.8)
        still_his = sum(1 for _, aa, _, _ in g if aa == "H")
        log("%-14s i_ptm %.3f (stage3 %.3f)  epitope %d, %d of %d parent (%.0f%%), "
            "%d on site | His kept %d/%d, facing partner %d/%d, Cb %s"
            % (sw["name"], row["sw"]["i_ptm"], sw["i_ptm"], len(row["sw"]["epitope"]),
               row["sw"]["kept_parent"], len(parent_epi),
               100*row["sw"]["frac_parent"], len(row["sw"]["on_site"]),
               still_his, len(g), near, len(g),
               ", ".join("%s%d->%s %.1f" % (aa, p, who, d) for p, aa, who, d in g)))
        if "ctl" in row:
            log("   %-11s control i_ptm %.3f  epitope %d, %d of %d parent (%.0f%%)"
                % (sw["source_backbone"], row["ctl"]["i_ptm"],
                   len(row["ctl"]["epitope"]), row["ctl"]["kept_parent"],
                   len(parent_epi), 100*row["ctl"]["frac_parent"]))
    except Exception as e:
        row["error"] = "%s %s" % (type(e).__name__, str(e)[:200])
        log("%s FAILED %s" % (sw["name"], row["error"]))
    with open(rec_path, "a") as fh:
        fh.write(json.dumps(row) + chr(10))
log("worker done")
"""

with open("/kaggle/working/confirm2_worker.py", "w") as fh:
    fh.write(textwrap.dedent(WORKER))

procs = []
for shard, gpu in enumerate(["0", "1"]):
    p = subprocess.Popen([sys.executable, "-u", "/kaggle/working/confirm2_worker.py",
                          gpu, str(shard), str(TIME_BUDGET_MIN)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    procs.append(p)
    print("launched confirm worker %d on GPU %s" % (shard, gpu), flush=True)

import threading
def pump(p):
    for line in p.stdout:
        print(line.rstrip(), flush=True)
threads = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs]
[t.start() for t in threads]
for p in procs:
    p.wait()
[t.join(timeout=10) for t in threads]

rows = []
for f in sorted(glob.glob("/kaggle/working/confirm2_w*.jsonl")):
    for line in open(f):
        line = line.strip()
        if line:
            rows.append(json.loads(line))
rows.sort(key=lambda r: -r.get("sw", {}).get("i_ptm", 0))
json.dump(rows, open("/kaggle/working/tnf_confirm2.json", "w"), indent=1)

print("\n==== CONFIRMATION OF THE 8 SUBMITTED SWITCHES ====")
print("%-14s %7s %7s %8s %8s %6s %6s %-5s"
      % ("design", "i_ptm", "stage3", "epi kept", "ctl kept", "site", "His", "faces"))
for r in rows:
    if "sw" not in r:
        print("%-14s  ERROR %s" % (r["name"], r.get("error", "")))
        continue
    g = r.get("his_geometry", [])
    print("%-14s %7.3f %7.3f %7.0f%% %7s %6d %5d/%d %5d"
          % (r["name"], r["sw"]["i_ptm"], r["stage3_i_ptm"],
             100*r["sw"]["frac_parent"],
             ("%.0f%%" % (100*r["ctl"]["frac_parent"])) if "ctl" in r else "-",
             len(r["sw"]["on_site"]),
             sum(1 for _, aa, _, _ in g if aa == "H"), len(g),
             sum(1 for _, _, _, d in g if d <= 8.8)))

ok = [r for r in rows if "sw" in r and r["sw"]["frac_parent"] >= 0.60
      and len(r["sw"]["on_site"]) >= 5]
geo = [r for r in rows if "sw" in r
       and all(d <= 8.8 for _, _, _, d in r.get("his_geometry", []))]
print("\n%d of %d hold >=60%% of the parent epitope AND 5+ on-site contacts"
      % (len(ok), len(rows)))
print("%d of %d have EVERY designed histidine still within reach of its cation"
      % (len(geo), len(rows)))
print("\n===== COPY FROM HERE =====")
for r in rows:
    print(json.dumps(r, separators=(",", ":")))
print("===== COPY TO HERE =====")